# EDA + storytelling — Performance d'un restaurant

## Objectif métier

Transformer une EDA en recommandations utilisables par un Product Manager / responsable d'activité.

**Dataset choisi :** `tips`, un jeu de données de transactions de restaurant disponible directement dans Plotly Express.

Le dataset contient le montant de l'addition, le pourboire, le sexe du client, le statut fumeur, le jour, le moment du repas et la taille du groupe.

> **Important :** il s'agit d'un petit dataset de démonstration (244 transactions). Les recommandations sont donc des pistes de décision à tester et non des conclusions généralisables à tous les restaurants.


## 1. Questions métier

Avant d'explorer les données, je me pose 5 questions :

1. Où se concentre le chiffre d'affaires : quels jours sont les plus importants ?
2. Le dîner est-il réellement le créneau prioritaire ?
3. Les grandes tables génèrent-elles des tickets plus élevés ?
4. Le montant de l'addition est-il lié au pourboire ?
5. Existe-t-il des anomalies ou des problèmes de qualité pouvant fausser les KPI ?

L'objectif est de transformer les réponses en recommandations actionnables.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")

df = px.data.tips().copy()
df["tip_pct"] = df["tip"] / df["total_bill"] * 100

print("Dimensions :", df.shape)
display(df.head())


## 2. Qualité des données

Je vérifie les types, les valeurs manquantes et les doublons avant de calculer les KPI.


In [ ]:
print("Valeurs manquantes :")
display(df.isna().sum())

print("Nombre de doublons :", df.duplicated().sum())

print("\nStatistiques :")
display(df[["total_bill", "tip", "size", "tip_pct"]].describe().round(2))

df_clean = df.drop_duplicates().copy()
print("\nTransactions après déduplication :", len(df_clean))


### Conclusion

Les données ne contiennent aucune valeur manquante. Un doublon exact est présent et sera retiré avant les analyses métier pour éviter de compter deux fois la même transaction.

## 3. Question 1 — Où se concentre le chiffre d'affaires ?

**Analyse :** comparaison du volume de transactions et du chiffre d'affaires par jour.


In [ ]:
day_order = ["Thur", "Fri", "Sat", "Sun"]

by_day = df_clean.groupby("day").agg(
    transactions=("total_bill", "size"),
    revenue=("total_bill", "sum"),
    avg_ticket=("total_bill", "mean"),
    avg_tip_pct=("tip_pct", "mean")
).reindex(day_order)

display(by_day.round(2))

by_day["revenue"].plot(kind="bar", figsize=(8, 4), title="Chiffre d'affaires par jour")
plt.xlabel("Jour")
plt.ylabel("Chiffre d'affaires")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Conclusion

Le samedi génère le plus gros chiffre d'affaires (**1 778,40**) avec **87 transactions**. Le dimanche a moins de volume mais le ticket moyen le plus élevé (**21,41**). **Recommandation : sécuriser en priorité la capacité du samedi et réserver les tests d'upsell au dimanche.**

## 4. Question 2 — Le dîner est-il le créneau prioritaire ?

**Analyse :** comparaison déjeuner / dîner sur le volume, le chiffre d'affaires et le ticket moyen.


In [ ]:
time_order = ["Lunch", "Dinner"]

by_time = df_clean.groupby("time").agg(
    transactions=("total_bill", "size"),
    revenue=("total_bill", "sum"),
    avg_ticket=("total_bill", "mean"),
    avg_tip_pct=("tip_pct", "mean")
).reindex(time_order)

display(by_time.round(2))

by_time["revenue"].plot(kind="bar", figsize=(7, 4), title="Chiffre d'affaires par créneau")
plt.xlabel("Créneau")
plt.ylabel("Chiffre d'affaires")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Conclusion

Le dîner représente **3 660,30** de chiffre d'affaires contre **1 167,47** pour le déjeuner, avec un ticket moyen de **20,80** contre **17,17**. **Recommandation : conserver le dîner comme priorité commerciale et utiliser le déjeuner comme terrain de test pour une offre d'upsell.**

## 5. Question 3 — Les grandes tables génèrent-elles des tickets plus élevés ?

**Analyse :** relation entre taille du groupe et montant de l'addition.


In [ ]:
by_size = df_clean.groupby("size").agg(
    transactions=("total_bill", "size"),
    revenue=("total_bill", "sum"),
    avg_ticket=("total_bill", "mean"),
    avg_tip_pct=("tip_pct", "mean")
)

display(by_size.round(2))

plt.figure(figsize=(8, 4))
sns.boxplot(data=df_clean, x="size", y="total_bill")
plt.title("Montant de l'addition selon la taille du groupe")
plt.xlabel("Nombre de personnes")
plt.ylabel("Addition")
plt.tight_layout()
plt.show()


### Conclusion

Le ticket moyen passe d'environ **16,45 pour 2 personnes** à **23,28 pour 3** et **28,61 pour 4**. **Recommandation : tester une formule de partage ou un menu groupe pour 3–4 personnes**, en vérifiant que le temps de service reste acceptable.

## 6. Question 4 — Le montant de l'addition est-il lié au pourboire ?

**Analyse :** nuage de points et corrélation entre l'addition et le pourboire.


In [ ]:
correlation = df_clean["total_bill"].corr(df_clean["tip"])
print(f"Corrélation addition / pourboire : {correlation:.2f}")

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x="total_bill", y="tip", hue="time")
plt.title("Addition et pourboire")
plt.xlabel("Addition")
plt.ylabel("Pourboire")
plt.tight_layout()
plt.show()


### Conclusion

La corrélation est d'environ **0,68** : les additions plus élevées sont généralement associées à des pourboires plus élevés. **Recommandation : travailler l'augmentation du ticket moyen, tout en évaluant séparément la marge et le taux de pourboire car corrélation ne signifie pas causalité.**

## 7. Question 5 — Existe-t-il des anomalies ou problèmes de qualité ?

**Analyse :** détection des valeurs atypiques avec la règle des 1,5 × IQR.


In [ ]:
def iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return int(mask.sum()), lower, upper

for col in ["total_bill", "tip_pct"]:
    n, lower, upper = iqr_outliers(df_clean[col])
    print(f"{col}: {n} valeurs atypiques | bornes IQR [{lower:.2f}, {upper:.2f}]")

plt.figure(figsize=(8, 4))
sns.boxplot(data=df_clean[["total_bill", "tip_pct"]])
plt.title("Valeurs atypiques")
plt.tight_layout()
plt.show()


### Conclusion

Le pourcentage de pourboire contient quelques valeurs atypiques, notamment lorsque l'addition est faible. **Recommandation : conserver ces lignes mais surveiller les KPI sensibles aux extrêmes et contrôler les transactions atypiques avant de comparer les performances.**

## 8. Corrélations complémentaires



In [ ]:
corr = df_clean[["total_bill", "tip", "size", "tip_pct"]].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", center=0)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

display(corr.round(2))


### Conclusion

La taille du groupe est positivement liée au montant de l'addition (**0,60**) et l'addition au pourboire (**0,68**). Ces relations suggèrent que le **nombre de convives et le ticket** sont deux leviers utiles à suivre ensemble dans les KPI métier.

# 9. Synthèse — 4 recommandations actionnables

### 1. Sécuriser le samedi
Renforcer staffing, disponibilité des tables et gestion des temps d'attente sur le samedi, qui concentre le plus gros volume et chiffre d'affaires.

### 2. Tester l'upsell le dimanche
Profiter du ticket moyen élevé du dimanche pour tester une proposition dessert/boisson/accompagnement.

### 3. Créer une offre pour les groupes de 3–4
Tester une formule de partage ou un menu groupe, car ces tailles de tables ont un ticket moyen nettement supérieur.

### 4. Expérimenter au déjeuner
Tester une formule simple pour augmenter le ticket moyen du déjeuner, puis mesurer l'effet avant/après.

**KPI de suivi :** chiffre d'affaires, ticket moyen, nombre de transactions, temps d'attente et taux d'adoption des offres.


# Conclusion

L'analyse ne doit pas s'arrêter aux constats. Les données suggèrent quatre actions prioritaires : protéger le samedi, développer l'upsell du dimanche, mieux monétiser les groupes de 3–4 et expérimenter au déjeuner.

La prochaine étape consiste à valider ces hypothèses sur plusieurs semaines de données réelles avant de généraliser les décisions.
